In [1]:
import torch
from monai.data import ImageDataset, DataLoader
from monai.transforms import EnsureChannelFirst, Compose, Rand3DElastic, Resize, ScaleIntensity, RandShiftIntensity
import monai
from torch.utils.tensorboard import SummaryWriter
import json
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Transforms from the MONAI tutorial
train_transforms = Compose([ScaleIntensity(), 
                            EnsureChannelFirst(), 
                            Resize((96, 96, 96)),
                            Rand3DElastic(prob=0.5, sigma_range=(5, 7), magnitude_range=(50, 150),spatial_size=(96, 96, 96), padding_mode="border"),
                            RandShiftIntensity(prob=0.5, offsets=0.10)
])

val_transforms = Compose([ScaleIntensity(), EnsureChannelFirst(), Resize((96, 96, 96))])

## 1. Load the locked-in splits
load_path = os.path.expanduser('~/Desktop/brain-math/deeplearn/GLM/GLM_kfold_splits.json')
with open(load_path, 'r') as f:
    saved_splits = json.load(f)

# 2. Select which fold you want to train right now
current_fold = "fold_1"  # Change this to "fold_2", "fold_3", etc., when ready
print(f"Loading data for {current_fold}...")

fold_data = saved_splits[current_fold]
train_images = fold_data["train_images"]
val_images = fold_data["val_images"]

# 3. Convert the saved integer labels (0 or 1) back into one-hot tensors for MONAI
train_labels = torch.nn.functional.one_hot(torch.as_tensor(fold_data["train_labels"])).float()
val_labels = torch.nn.functional.one_hot(torch.as_tensor(fold_data["val_labels"])).float()

# 4. Create your MONAI Datasets
train_ds = ImageDataset(image_files=train_images, labels=train_labels, transform=train_transforms)
val_ds = ImageDataset(image_files=val_images, labels=val_labels, transform=val_transforms)

# 5. Create DataLoaders
train_loader = DataLoader(train_ds, batch_size=2, shuffle=True, num_workers=2, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_ds, batch_size=2, num_workers=2, pin_memory=torch.cuda.is_available())

# Create UNet, CrossEntropyLoss and Adam optimizer
model = monai.networks.nets.ResNet(
    block="basic",         
    layers=[2, 2, 2, 2],   
    block_inplanes=[64, 128, 256, 512],
    spatial_dims=3,
    n_input_channels=1,
    num_classes=2
)

loss_function = torch.nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), 1e-4)

# start a typical PyTorch training
best_metric = -1
best_metric_epoch = -1
epoch_loss_values = []
metric_values = []
writer = SummaryWriter()
max_epochs = 150

patience = 10            # Stop after 10 validation checks without improvement
patience_counter = 0     # Tracks how long we've gone without a new best score

for epoch in range(max_epochs):
    print("-" * 10)
    print(f"epoch {epoch + 1}/{max_epochs}")
    model.train()
    epoch_loss = 0
    step = 0

    for batch_data in train_loader:
        step += 1
        inputs, labels = batch_data[0].to(device), batch_data[1].to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        epoch_len = len(train_ds) // train_loader.batch_size
        print(f"{step}/{epoch_len}, train_loss: {loss.item():.4f}")
        writer.add_scalar("train_loss", loss.item(), epoch_len * epoch + step)

    epoch_loss /= step
    epoch_loss_values.append(epoch_loss)
    print(f"epoch {epoch + 1} average loss: {epoch_loss:.4f}")


    model.eval()

    num_correct = 0.0
    metric_count = 0
    for val_data in val_loader:
        val_images, val_labels = val_data[0].to(device), val_data[1].to(device)
        with torch.no_grad():
            val_outputs = model(val_images)
            value = torch.eq(val_outputs.argmax(dim=1), val_labels.argmax(dim=1))
            metric_count += len(value)
            num_correct += value.sum().item()

    metric = num_correct / metric_count
    metric_values.append(metric)

    if metric > best_metric:
        best_metric = metric
        best_metric_epoch = epoch + 1
        patience_counter = 0  # reset patience if it improves
        torch.save(model.state_dict(), "best_resnet_classification3d_array.pth")
        print("saved new best metric model")
    else:
        patience_counter += 1 # increment patience if it failed to improve
        print(f"No improvement. Patience: {patience_counter}/{patience}")

    print(f"Current epoch: {epoch+1} current accuracy: {metric:.4f} ")
    print(f"Best accuracy: {best_metric:.4f} at epoch {best_metric_epoch}")
    writer.add_scalar("val_accuracy", metric, epoch + 1)

    if patience_counter >= patience:
        print(f"\nEarly stopping triggered at epoch {epoch + 1}!")
        print(f"Validation accuracy hasn't improved in {patience} epochs.")
        break

print(f"Training completed, best_metric: {best_metric:.4f} at epoch: {best_metric_epoch}")
writer.close()

Loading data for fold_1...
----------
epoch 1/150
1/96, train_loss: 0.9357
2/96, train_loss: 0.4739
3/96, train_loss: 1.5786
4/96, train_loss: 0.8084
5/96, train_loss: 0.5266
6/96, train_loss: 0.5534
7/96, train_loss: 1.2174
8/96, train_loss: 0.9950
9/96, train_loss: 0.5195
10/96, train_loss: 0.9026
11/96, train_loss: 0.9175
12/96, train_loss: 0.8578
13/96, train_loss: 0.6893
14/96, train_loss: 0.7624
15/96, train_loss: 0.5408
16/96, train_loss: 0.5787
17/96, train_loss: 0.5932
18/96, train_loss: 1.1001
19/96, train_loss: 0.3083
20/96, train_loss: 0.7010
21/96, train_loss: 1.4993
22/96, train_loss: 1.2376
23/96, train_loss: 0.8580
24/96, train_loss: 1.0909
25/96, train_loss: 0.7732
26/96, train_loss: 0.6885
27/96, train_loss: 0.7795
28/96, train_loss: 0.7482
29/96, train_loss: 0.7693
30/96, train_loss: 0.7436
31/96, train_loss: 0.5994
32/96, train_loss: 0.8823
33/96, train_loss: 0.5486
34/96, train_loss: 0.5210
35/96, train_loss: 0.6025
36/96, train_loss: 1.1026
37/96, train_loss: 0.43

KeyboardInterrupt: 